In [ ]:
%cd ../..
import os
import torch
import polars as pl
import pydicom
import numpy as np
from tqdm import tqdm
from omegaconf import OmegaConf
import matplotlib.pyplot as plt
from einops import rearrange

import torch.nn.functional as F
from sklearn.decomposition import PCA

from dinov2.inference import build_model, view_volume, load_dicom, crop_volume

In [ ]:
def get_metadata(data_path):
    metadata_raw = pl.read_csv(os.path.join(data_path, "NSCLC-Radiomics/metadata.csv"))
    metadata_raw = metadata_raw.select(["Subject ID", "Modality", "File Location"])

    ct_meta = metadata_raw.filter(pl.col("Modality") == "CT").drop("Modality")
    seg_meta = metadata_raw.filter(pl.col("Modality") == "SEG").drop("Modality")

    merged = ct_meta.join(seg_meta, on="Subject ID", how="inner", suffix="_SEG")

    merged = merged.rename(
        {"File Location": "CT File Location", "File Location_SEG": "SEG File Location"}
    )
    return merged

In [ ]:
data_path = "/scratch/VM/radio-foundation/datasets"

metadata = get_metadata(data_path)
metadata.head()

In [ ]:
def get_bbox(folder_path: str, label: str, pad=0):
    files = [
        os.path.join(folder_path, x)
        for x in os.listdir(folder_path)
        if x.endswith(".dcm")
    ]
    if len(files) != 1:
        raise RuntimeError(f"Expected to find exactly one file. Found: {len(files)}")

    ds = pydicom.dcmread(files[0])

    segments = ds.SegmentSequence
    per_frame = ds.PerFrameFunctionalGroupsSequence

    n_frames = ds.NumberOfFrames
    rows = ds.Rows
    cols = ds.Columns

    pixel_array = np.frombuffer(ds.PixelData, dtype=np.uint8)
    pixel_array = np.unpackbits(pixel_array)
    pixel_array = pixel_array[: n_frames * rows * cols]
    pixel_array = pixel_array.reshape((n_frames, rows, cols))

    frame_segment_numbers = [
        f.SegmentIdentificationSequence[0].ReferencedSegmentNumber for f in per_frame
    ]

    lung_mask = None

    for seg in segments:
        if label in seg.SegmentLabel:
            seg_num = seg.SegmentNumber
            seg_frames_idx = [
                i for i, s_num in enumerate(frame_segment_numbers) if s_num == seg_num
            ]

            if lung_mask is None:

                lung_mask = pixel_array[seg_frames_idx]

            else:

                lung_mask += pixel_array[seg_frames_idx]

    lung_mask = torch.from_numpy(lung_mask > 0) # type: ignore

    D, H, W = lung_mask.shape

    coords = lung_mask.nonzero(as_tuple=False)
    if coords.numel() == 0:
        raise ValueError("No bounding box to extract.")

    z_min, y_min, x_min = coords.min(dim=0).values
    z_max, y_max, x_max = coords.max(dim=0).values

    def pad_dim(vmin, vmax, pad_val, r_lim):
        lpad = pad_val//2
        rpad = pad_val - lpad
        vmin = max(0, vmin - lpad)
        vmax = min(r_lim, vmax + rpad)

        return vmin, vmax
    
    z_min, z_max = pad_dim(z_min, z_max, pad, D)
    y_min, y_max = pad_dim(y_min, y_max, pad, H)
    x_min, x_max = pad_dim(x_min, x_max, pad, W)
    
    new_H = y_max - y_min
    new_W = x_max - x_min
    H_pad = max(new_H, new_W) - new_H
    W_pad = max(new_H, new_W) - new_W

    y_min, y_max = pad_dim(y_min, y_max, H_pad, H)
    x_min, x_max = pad_dim(x_min, x_max, W_pad, W)

    bbox = (slice(z_min, z_max + 1), slice(y_min, y_max + 1), slice(x_min, x_max + 1))

    return bbox


In [ ]:
config_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/config.yaml"
checkpoint_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/eval/training_79999/teacher_checkpoint.pth"

device = torch.device("cuda")

config = OmegaConf.load(config_path)
model, autocast_ctx = build_model(checkpoint_path, config, img_size=504, device=device)

In [ ]:
patient_id, study_path, rel_seg_path = metadata.row(0)
dcm_path = os.path.join(data_path, study_path)
seg_path = os.path.join(data_path, rel_seg_path)
img, spacing = load_dicom(dcm_path)

tumour_bbox = get_bbox(seg_path, label="Neoplasm", pad=50)

cropped_img = img[tumour_bbox]

view_volume(cropped_img, spacing)
print(img.shape, cropped_img.shape)

In [ ]:
img_size = 224
patch_size = 14
patch_dim = img_size // patch_size

test_img = cropped_img[30:40]
test_img = test_img.unsqueeze(0)
test_img = torch.nn.functional.interpolate(
    test_img, size=(img_size, img_size), mode="bilinear"
)

with torch.inference_mode():
    with autocast_ctx():
        features = model.forward_features(test_img.cuda())
features = {k: v.cpu() for k, v in features.items() if isinstance(v, torch.Tensor)}
patch_features = features["x_norm_patchtokens"]
patch_features = rearrange(patch_features, "1 (x y) d -> x y d", x=patch_dim, y=patch_dim)
patch_features.shape

In [ ]:
def plot_patch_similarity(patch_features, ref_image, ref_x, ref_y):
    ref_patch = patch_features[ref_y,ref_x].unsqueeze(0)
    img_size = ref_image.shape[-1]
    patch_dim = patch_features.shape[0]

    flat_features = rearrange(patch_features, "x y d -> (x y) d")
    
    cos_similarity = F.cosine_similarity(flat_features, ref_patch, dim=1)
    cos_similarity = rearrange(cos_similarity, "(x y) -> x y", x=patch_dim, y=patch_dim)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    im1 = ax1.imshow(ref_image[0,5], cmap='gray')
    ax1.plot(
        [
            (ref_x + 0.5) / patch_dim * (img_size - 1),
            (ref_x + 0.5) / patch_dim * (img_size - 1)
        ],
        [
            ref_y / patch_dim * (img_size - 1),
            (ref_y + 1) / patch_dim * (img_size - 1)
        ], c="r"
    )
    ax1.plot(
        [
            (ref_x) / patch_dim * (img_size - 1),
            (ref_x + 1) / patch_dim * (img_size - 1)
        ],
        [
            (ref_y + 0.5) / patch_dim * (img_size - 1),
            (ref_y + 0.5) / patch_dim * (img_size - 1)
        ], c="r"
    )
    fig.colorbar(im1, ax=ax1)
    ax1.set_title('CT Image')

    im2 = ax2.imshow(cos_similarity**2, cmap='plasma')
    fig.colorbar(im2, ax=ax2)
    ax2.set_title('Cosine Similarity')

    plt.tight_layout()
    plt.show()

In [ ]:
def pca_feature_map(features, components=(0,1,2)):
    H, W, D = features.shape
    X = features.view(H*W, D).numpy()

    max_comp = max(components) + 1

    pca = PCA(n_components=max_comp)
    X_pca = pca.fit_transform(X)

    X_sel = X_pca[:, components]

    X_min, X_max = X_sel.min(0), X_sel.max(0)
    X_sel = (X_sel - X_min) / (X_max - X_min + 1e-6)

    feature_map = X_sel.reshape(H, W, 3)

    return torch.tensor(feature_map, dtype=torch.float32)

In [ ]:
plot_patch_similarity(patch_features, test_img, ref_x = 9, ref_y = 6)

In [ ]:
pca_img = pca_feature_map(patch_features, (0,1,2))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

im1 = ax1.imshow(test_img[0,5], cmap='gray')
fig.colorbar(im1, ax=ax1)
ax1.set_title('CT Image')

im2 = ax2.imshow(pca_img)
fig.colorbar(im2, ax=ax2)
ax2.set_title('PCA')

plt.tight_layout()
plt.show()